In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import sys
from pathlib import Path

import jax
from jax import Array
from jax.nn import sigmoid
import jax.numpy as jnp
from jax import jit
from jax import config
config.update("jax_enable_x64", True)
config.update("jax_platform_name", "cpu")

import jaxley as jx

from jaxley.channels import Channel, Leak, Na, K
from jaxley.connect import connect
from jaxley.synapses.synapse import Synapse
from jaxley.solver_gate import exponential_euler

from math import pi
import matplotlib as mpl
import matplotlib.pyplot as plt

import numpy as np

import pandas as pd
import plotly.graph_objects as go
import plotly.io as pio
import pickle

from scipy.optimize import curve_fit

from typing import Callable, Dict, Optional

import warnings
warnings.filterwarnings("ignore", category=pd.errors.PerformanceWarning)

import pathlib
%matplotlib inline
%config InlineBackend.figure_format='retina'

inputpath_data = pathlib.Path.cwd() / 'input_data' 
inputpath_data.mkdir(parents=True, exist_ok=True)
savepath_data = pathlib.Path.cwd() / 'output_data' 
savepath_data.mkdir(parents=True, exist_ok=True)
savepath_fig = pathlib.Path.cwd() / 'output_fig' 
savepath_fig.mkdir(parents=True, exist_ok=True)

style_file = pathlib.Path.cwd().parent / "plot_style.mplstyle"
plt.style.use(["default", style_file])


# Model the voltage-gated component of Rdl

In [ ]:
# Fit the steady-state voltage-gated component of Rdl
iv = pd.read_csv(inputpath_data / 'ephys_rdl_IV_steady_state_current_nA.csv')
v_data = iv['holding_potential_mV'].values
i_data = iv['steady_state_current_nA'].values

e_rev = -70.0
valid = v_data != e_rev  # Avoid division-by-zero

# Calculate conductance
g_data = i_data[valid] / (v_data[valid] - e_rev)

def boltzmann_model(v, g_min, g_amp, v_half, k):
    return g_min + g_amp / (1.0 + np.exp(-(v - v_half) / k))

popt, _ = curve_fit(boltzmann_model, v_data[valid], g_data, p0=[0.001, 0.01, 0.0, 20.0])
g_min, g_amp, v_half, k = popt
f_min = g_min / (g_min + g_amp)

print(f"v_half = {v_half:.2f} mV")
print(f"k = {k:.2f} mV")
print(f"f_min = {f_min:.4f}")

In [ ]:
# Fit the activation kinetics of the voltage-gated component of Rdl
trace = pd.read_csv(inputpath_data / 'rdl_40mV_current_response_nA_no_capacitive.csv')
trace_t = trace['time_from_vstep'].values * 1000   # s -> ms
trace_i = trace['current_nA'].values / 1000        # pA -> nA

# Restrict fit window (ms)
fit_start_time = 1.0    # Skip the capacitive artifact
fit_end_time = 100.0    # Truncate the long steady-state tail

fit_mask = (trace_t >= fit_start_time) & (trace_t <= fit_end_time)
t_fit = trace_t[fit_mask]
i_fit = trace_i[fit_mask]

# Fit exponential anchored at I(0) = 0
def exp_fit_func(t, I_ss, tau):
    return I_ss * (1.0 - np.exp(-t / tau))

# Initial guesses
I_ss_guess = np.mean(i_fit[-int(len(i_fit)*0.2):])  # Mean of the end of the 200-ms window
tau_guess = 5.0                                     
p0 = [I_ss_guess, tau_guess]

# Bounds
min_val = np.min(i_fit)
max_val = np.max(i_fit)
bounds = (
    [min_val - 1.0, 0.1],       # Lower bounds [I_ss, tau]
    [max_val + 1.0, 1000.0]     # Upper bounds [I_ss, tau]
)

popt, pcov = curve_fit(exp_fit_func, t_fit, i_fit, p0=p0, bounds=bounds)
I_ss_fit, tau_fit = popt

print(f"Fit for window from {fit_start_time} to {fit_end_time} ms:")
print(f"  Steady-state current: {I_ss_fit:.4f} nA")
print(f"  Activation time constant at +40 mV: {tau_fit:.4f} ms")

# Plot
fig, ax = plt.subplots(figsize=(5, 3))

# Raw data
ax.plot(trace_t, trace_i, 'k.', label='Data', alpha=0.2)

# Fit
t_plot = np.linspace(0, fit_end_time, 400)
fit_curve_y = exp_fit_func(t_plot, *popt)
ax.plot(t_plot, fit_curve_y, 'r-', label=f'Fit (τ = {tau_fit:.2f} ms)', lw=2)

# Mark window
ax.axvline(x=fit_start_time, color='gray', linestyle='--', alpha=0.5)
ax.axvline(x=fit_end_time, color='gray', linestyle='--', alpha=0.5, label='Fit window')

ax.set_xlabel('Time (ms)')
ax.set_ylabel('Current (nA)')
ax.set_xlim(-5, fit_end_time + 20)
ax.legend(frameon=False)
ax.spines[['right', 'top']].set_visible(False)

In [ ]:
# Model Rdl as voltage-gated channel
class Rdl(Channel):
    """Chloride channel with a time-dependent Hodgkin-Huxley-type activation gate"""
    def __init__(self, name=None):
        self.current_is_in_mA_per_cm2 = True
        super().__init__(name)
        
        self.channel_params = {
            "rdl_g": 0.0,
            "rdl_e": -70.0,
            "rdl_v_half": 48.17,
            "rdl_k": 32.57,
            "rdl_f_min": 0.0496,
            "rdl_tau": 1.6437        
        }
        self.channel_states = {
            "rdl_m": 0.0          
        }
        self.current_name = "i_rdl"

    def update_states(self, states, dt, v, params):  
        m_old = states["rdl_m"]
        v_half = params["rdl_v_half"]
        k = params["rdl_k"]
        tau = params["rdl_tau"]
        
        m_inf = 1.0 / (1.0 + jnp.exp(-(v - v_half) / k))
        new_m = m_inf + (m_old - m_inf) * jnp.exp(-dt / tau)
        return {"rdl_m": new_m}

    def compute_current(self, states, v, params): 
        g = params["rdl_g"]
        e = params["rdl_e"]
        f_min = params["rdl_f_min"]
        m = states["rdl_m"]
        
        g_eff = g * (f_min + (1.0 - f_min) * m)
        return g_eff * (v - e)

    def init_state(self, states, v, params, dt):
        v_half = params["rdl_v_half"]
        k = params["rdl_k"]
        m_inf = 1.0 / (1.0 + jnp.exp(-(v - v_half) / k))
        return {"rdl_m": m_inf}

In [ ]:
# Simulate voltage-clamp experiment
def run_rdl_voltage_clamp():
    # Simulation timing
    dt = 0.2        # ms
    t_max = 1000.0   # ms
    num_steps = int(t_max / dt) + 1
    t_span = np.linspace(0.0, t_max, num_steps)
    
    # Voltage-clamp protocol
    t_start, t_end = 250.0, 750.0
    voltage_steps = np.arange(-120, 41, 10)  # -120 to +40 mV in 10 mV
    
    current_traces = {}
    steady_state_currents = []
    
    for v_step in voltage_steps:
        # Re-initialize the compartment to avoid carrying over clamp states
        comp = jx.Compartment()
        comp.insert(Rdl())
        
        # Set conductances (S/cm^2) and reversal potentials (mV)
        comp.set("rdl_g", 0.005)  
        comp.set("rdl_e", -70.0)
        
        # Create command voltage trace
        v_command = np.full_like(t_span, -70.0)
        v_command[(t_span >= t_start) & (t_span < t_end)] = v_step
        
        # Set holding state voltage
        comp.set("v", -70.0)
        
        comp.clamp("v", v_command)
        comp.delete_recordings()  # Clear previous recordings to avoid accumulation in loops
        comp.record("i_rdl")
        
        # Run
        sim_data = jx.integrate(comp, t_max=t_max, delta_t=dt)
        
        i_trace = np.array(sim_data).squeeze()
        current_traces[v_step] = i_trace
        
        # Extract steady-state current at the end of the voltage step
        ss_idx = int(t_end / dt) - 1
        steady_state_currents.append(i_trace[ss_idx])
        
    return t_span, voltage_steps, current_traces, steady_state_currents, t_start, t_end

# Run and plot
t_span, voltage_steps, current_traces, ss_currents, t_start, t_end = run_rdl_voltage_clamp()

fig, axes = plt.subplots(1, 3, figsize=(11, 3))
gaba_color = "#7570B3"

# Voltage steps
for v_step in voltage_steps:
    v_command = np.full_like(t_span, -70.0)
    v_command[(t_span >= t_start) & (t_span < t_end)] = v_step
    axes[0].plot(t_span, v_command, color="k", linewidth=1)
axes[0].set_xlabel("Time (ms)")
axes[0].set_ylabel("Membrane potential (mV)")

# Currents
for i, v_step in enumerate(voltage_steps):
    color_intensity = 0.3 + 0.7 * (i / len(voltage_steps))
    label = f"{v_step} mV" if v_step in [-120, -70, 0, 50] else ""
    axes[1].plot(t_span, current_traces[v_step], color=gaba_color, label=label)
    
axes[1].set_xlabel("Time (ms)")
axes[1].set_ylabel("Current density (mA/cm²)")

# Steady-state I-V
axes[2].plot(voltage_steps, ss_currents, 'o-', color=gaba_color, linewidth=2.5, markersize=7)
axes[2].axhline(0, color='k', linestyle='--', alpha=0.3)
axes[2].set_xlabel("Membrane potential (mV)")
axes[2].set_ylabel("Steady-state current (mA/cm²)")

for i in range(3):
    axes[i].spines[['right', 'top']].set_visible(False)

plt.tight_layout()

# Model Para

In [ ]:
def para_sigmoid(v, half_v, slope):
    return 1 / (1 + jnp.exp(-(v - half_v) / slope))
    
def bell(v, mean_val, max_val, min_val, std):
    return jnp.exp(-((v - mean_val) ** 2) / std**2) * (max_val - min_val) + min_val

tau_sigmoid_params = {
    'ParaReMeasured': {
        'act': jnp.asarray([-35.3411, 0.9264, 0.0808, 29.3799]),
        'inact': jnp.asarray([-35.9521, 0.5882, 0.314, 42.3669])},
}

class ParaReMeasured(Channel):
    """Para sodium channel."""

    def __init__(self, name=None):
        self.current_is_in_mA_per_cm2 = True
        super().__init__(name)
        self.channel_params = {
            "Para_g": 1e-5,
            "ENa": 50.0,
            # Sigmoid for the time constants of activation.
            "Para_m_tau_mean": tau_sigmoid_params["ParaReMeasured"]["act"][0],
            "Para_m_tau_min_val": tau_sigmoid_params["ParaReMeasured"]["act"][2],
            "Para_m_tau_max_val": tau_sigmoid_params["ParaReMeasured"]["act"][1],
            "Para_m_tau_std": tau_sigmoid_params["ParaReMeasured"]["act"][3],
            # Sigmoid for the stead-state of activation.
            "Para_m_inf_half_v": -49.06,
            "Para_m_inf_slope": 4.42,
            # Sigmoid for the time constants of inactivation.
            "Para_h_tau_mean": tau_sigmoid_params["ParaReMeasured"]["inact"][0],
            "Para_h_tau_min_val": tau_sigmoid_params["ParaReMeasured"]["inact"][2],
            "Para_h_tau_max_val": tau_sigmoid_params["ParaReMeasured"]["inact"][1],
            "Para_h_tau_std": tau_sigmoid_params["ParaReMeasured"]["inact"][3],
            # Sigmoid for the stead-state of inactivation.
            "Para_h_inf_half_v": -41.10,
            "Para_h_inf_slope": -6.20,
        }
        self.channel_states = {"Para_m": 0.0, "Para_h": 0.0}
        self.current_name = "i_Na"

    def update_states(self, states, dt, v, params):
        """Update state."""
        m_inf = para_sigmoid(
            v,
            params["Para_m_inf_half_v"],
            params["Para_m_inf_slope"],
        )
        m_tau = bell(
            v,
            params["Para_m_tau_mean"],
            params["Para_m_tau_max_val"],
            params["Para_m_tau_min_val"],
            params["Para_m_tau_std"],
        )

        h_inf = para_sigmoid(
            v,
            params["Para_h_inf_half_v"],
            params["Para_h_inf_slope"],
        )
        h_tau = bell(
            v,
            params["Para_h_tau_mean"],
            params["Para_h_tau_max_val"],
            params["Para_h_tau_min_val"],
            params["Para_h_tau_std"],
        )

        new_m = exponential_euler(states["Para_m"], dt, m_inf, m_tau)
        new_h = exponential_euler(states["Para_h"], dt, h_inf, h_tau)
        return {"Para_m": new_m, "Para_h": new_h}

    def compute_current(self, states, v, params):
        """Return current."""
        m = states["Para_m"]
        h = states["Para_h"]
        kd_conds = params["Para_g"] * m**3 * h  # S/cm^2
        return kd_conds * (v - params["ENa"])

    def init_state(self, states, v, params, delta_t):
        m_inf = para_sigmoid(
            v,
            params["Para_m_inf_half_v"],
            params["Para_m_inf_slope"],
        )
        h_inf = para_sigmoid(
            v,
            params["Para_h_inf_half_v"],
            params["Para_h_inf_slope"],
        )
        return {"Para_m": m_inf, "Para_h": h_inf}

# Conductance-based synapse

In [ ]:
class ConductanceSynapse(Synapse):
    def __init__(self, nonlinearity: Callable = sigmoid, name: Optional[str] = None):
        super().__init__(name)
        prefix = self._name
        self.synapse_params = {
            f"{prefix}_gS": 1e-4,  # uS
            f"{prefix}_v_th": -35.0,  # mV
            f"{prefix}_delta": 10.0,  # mV
            f"{prefix}_e_syn": 0.0,
        }
        self.synapse_states = {f"{prefix}_s": 0.0}
        self.nonlinearity = nonlinearity

    def update_states(
        self,
        states: Dict,
        delta_t: float,
        pre_voltage: float,
        post_voltage: float,
        params: Dict,
    ) -> Dict:
        """Return updated synapse state and current."""
        prefix = self._name
        activation = self.nonlinearity(
            (pre_voltage - params[f"{prefix}_v_th"])
            / params[f"{prefix}_delta"]
        )
        return {f"{prefix}_s": activation}

    def compute_current(
        self, states: Dict, pre_voltage: float, post_voltage: float, params: Dict
    ) -> float:
        prefix = self._name
        activation = states[f"{prefix}_s"]
        g_syn = params[f"{prefix}_gS"] * activation
        return g_syn * (post_voltage - params[f"{prefix}_e_syn"])

# Compartmental model

In [ ]:
# Path
fname = inputpath_data / 'T4_skeleton_full.swc' # FlyWire cell ID 720575940612221669

# SWC format
names = ['id', 'type', 'x', 'y', 'z', 'r', 'parent']

# Load morphology
df_final = pd.read_csv(fname, sep=r'\s+', comment='#', names=names)
df_final[['x', 'y', 'z', 'r']] *= 0.001 # Scale to μm (FlyWire SWC coordinates are in nm). 
df_final['r'] = df_final['r'].clip(lower=0.01) # Min radius of 0.01 μm

# Add soma
soma_node_id = 1308 
df_final.loc[df_final['id'] == soma_node_id, 'type'] = 1  

# Save μm SWC file
final_swc_um = savepath_data / 'T4_skeleton_full_um.swc'
df_final.to_csv(final_swc_um, sep=' ', header=False, index=False)

# Build cell
cell = jx.read_swc(final_swc_um, ncomp=1, min_radius=0.01) # Min radius of 0.01 μm
cell.set("axial_resistivity", 250.0) # Ω·cm
cell.set("capacitance", 1.0)         # μF/cm^2

# Assign subcellular compartments
cell.branch(range(52, 284)).add_to_group("dendrite")
cell.branch([0, 1, 2, 3, 4, 5, 284, 285]).add_to_group("soma_fiber")
cell.branch(6).add_to_group("axon")
cell.branch(range(7, 52)).add_to_group("axon_terminal")

t4_raw_synapses = pd.read_csv(inputpath_data / 'T4_synapse_info.csv')
syn_counts_per_branch = t4_raw_synapses['swc_node'].value_counts().to_dict()

# dλ rule to get the number of compartments per branch
frequency = 100.0
d_lambda = 0.1

for branch in cell.branches:
    diameter = 2 * branch.nodes["radius"].to_numpy()[0]
    c_m = branch.nodes["capacitance"].to_numpy()[0]
    r_a = branch.nodes["axial_resistivity"].to_numpy()[0]
    l = branch.nodes["length"].to_numpy()[0]

    lambda_f = 1e5 * np.sqrt(diameter / (4 * np.pi * frequency * c_m * r_a))
    ncomp = int((l / (d_lambda * lambda_f) + 0.9) / 2) * 2 + 1
    
    branch.set_ncomp(ncomp, min_radius=0.01, initialize=False)

# Initialize and scale soma
cell.initialize()
cell.soma.set("radius", 1.5 / np.sqrt(2))
cell.soma.set("length", 1.5 / np.sqrt(2))

# Rotate cell for display purposes
cell.rotate(20, rotation_axis="xz")
cell.rotate(-65, rotation_axis="xy")
cell.rotate(-20, rotation_axis="yz")


In [ ]:
# Visualise cell
fig, ax = plt.subplots(1, 1, figsize=(3, 3))
ax = cell.vis(ax=ax, dims=(0, 1))
ax = cell.soma.vis(ax=ax, color="k", zorder=1000)
ax = cell.soma_fiber.vis(ax=ax, color="#CC6677")
ax = cell.dendrite.vis(ax=ax, color="#44AA99")
ax = cell.axon.vis(ax=ax, color="#332288", dims=(0, 1))
ax = cell.axon_terminal.vis(ax=ax, color="#88CCEE")
_ = ax.set_aspect("equal")
ax.set_axis_off()

In [ ]:
# Set up passive and active parameters
cell.insert(Leak())
cell.set("v", -60.0)

cell.set("Leak_gLeak", 1 / 20_000) # Specific membrane resistivity (20 kΩ·cm^2)
cell.set("capacitance", 1.0) # Specific membrane capacitance of (1 µF/cm^)

# Insert voltage-gated Rdl component (but keep at 0 S for now)
cell.insert(Rdl())
cell.set("rdl_g", 0.0)

## Insert voltage-gated sodium channel (Vieira et al., 2026)
cell.insert(ParaReMeasured())
cell.set("Para_g", 0.0)
#cell.axon.set("Para_g", 0.0003)
cell.axon.set("Para_g", 0.0015)

cell.init_states()

In [ ]:
# Create trunks of medullary neurons
presyn_neuron = jx.Cell()
presyn_neuron.set("length", 100.0)
presyn_neuron.set("radius", 1.0)
presyn_neuron.insert(Leak())

In [ ]:
# Connect network
net = jx.Network([presyn_neuron, presyn_neuron, presyn_neuron, cell])

# Name cells
T4 = net.cell(3)
Mi9 = net.cell(0)
Mi1 = net.cell(1)
Mi4 = net.cell(2)

# Clear any old broken connections
if hasattr(net, 'edges') and net.edges is not None:
    net.edges = net.edges.iloc[0:0] 

# Load synapse CSV
synapse_df = pd.read_csv(inputpath_data / 'T4_synapse_info.csv')
synapse_df['type_clean'] = synapse_df['type'].str.strip().str.lower()

# Raw scaled coordinates (in current SWC coordinate frame)
syn_x = synapse_df['x'].to_numpy() * 0.001
syn_y = synapse_df['y'].to_numpy() * 0.001
syn_z = synapse_df['z'].to_numpy() * 0.001
syn_types = synapse_df['type_clean'].to_numpy()

# Consider only dendritic input synapses for mapping
dendrite_branches = set(range(52, 284))

# Build coordinate system
comp_coords = []
comp_references = []

for b_idx in dendrite_branches:
    try:
        branch_view = T4.branch(b_idx)
        branch_nodes = branch_view.nodes
        ncomp = branch_view.ncomp

        xs = branch_nodes["x"].to_numpy()
        ys = branch_nodes["y"].to_numpy()
        zs = branch_nodes["z"].to_numpy()

        if np.all(np.isnan(xs)):
            continue

    except (ValueError, KeyError, AttributeError):
        continue

    for c_idx in range(ncomp):
        node_idx = int((c_idx / ncomp) * len(branch_nodes))
        comp_coords.append([xs[node_idx], ys[node_idx], zs[node_idx]])
        comp_references.append((b_idx, c_idx))

comp_coords = np.array(comp_coords)

PHYSICAL_5UM_THRESHOLD = 5.0  

input_mapping = {
    "glutamate":     {"pre_cell_idx": 0, "count": 0},
    "acetylcholine": {"pre_cell_idx": 1, "count": 0},
    "gaba":          {"pre_cell_idx": 2, "count": 0}
}

wired_edge_indices_by_type = {"glutamate": [], "acetylcholine": [], "gaba": []}

# Map synapses
print("Mapping synapses...")

# Compartments with GABA input
gaba_compartments = set()

for syn_name, config in input_mapping.items():
    type_mask = (syn_types == syn_name)
    if not np.any(type_mask):
        continue

    s_x, s_y, s_z = syn_x[type_mask], syn_y[type_mask], syn_z[type_mask]
    pre_cell = net.cell(config["pre_cell_idx"])

    dx = s_x[:, None] - comp_coords[None, :, 0]
    dy = s_y[:, None] - comp_coords[None, :, 1]
    dz = s_z[:, None] - comp_coords[None, :, 2]

    dist_matrix = np.sqrt(dx**2 + dy**2 + dz**2)
    closest_comp_indices = np.argmin(dist_matrix, axis=1)
    min_distances = np.min(dist_matrix, axis=1)

    mapped_count = 0
    for i, comp_idx in enumerate(closest_comp_indices):
        if min_distances[i] <= PHYSICAL_5UM_THRESHOLD:
            local_b_idx, c_idx = comp_references[comp_idx]

            # Connect synapse if mapped to valid branch
            post_compartment = T4.branch(local_b_idx).comp(c_idx)
            start_idx = len(net.edges) if hasattr(net, 'edges') and net.edges is not None else 0

            connect(pre_cell, post_compartment, ConductanceSynapse())

            end_idx = len(net.edges)
            wired_edge_indices_by_type[syn_name].extend(range(start_idx, end_idx))
            config["count"] += (end_idx - start_idx)
            mapped_count += 1

            if syn_name == "gaba" and local_b_idx in dendrite_branches:
                gaba_compartments.add((local_b_idx, c_idx))


# To be tuned
# --------------------------------------------------------------------

# Overall maximal synaptic conductances (µS)
baseline_conductances = {
    "acetylcholine": 0.004,
    "glutamate":     0.008,
    "gaba":          0.015,
}

# Ligant-to-voltage-gated conductance ratio of Rdl
ratio = 1.2

# Convert gGABA (μS) to S and apply the ratio constraint
target_total_rdl_gS = (baseline_conductances["gaba"] * 1e-6) / ratio  

# --------------------------------------------------------------------

print("\nSingle-synapse conductances:")

for syn_name, edge_indices in wired_edge_indices_by_type.items():
    actual_count = len(edge_indices)
    
    if actual_count == 0:
        print(f"  {syn_name}: 0.000 μS (0 synapses found)")
        continue
        
    # Use raw µS values for Jaxley
    base_g_uS = baseline_conductances[syn_name]
    exact_single_g_uS = base_g_uS / actual_count
    
    reversal_potentials = {
        "acetylcholine": -21.0,
        "glutamate": -70.0,
        "gaba": -68.0
    }
    syn_e_syn = reversal_potentials[syn_name]
    
    print(f"  {syn_name}: {exact_single_g_uS:.8f} uS ({actual_count} synapses)")
    
    net.select(edges=edge_indices).set("ConductanceSynapse_gS", exact_single_g_uS)
    net.select(edges=edge_indices).set("ConductanceSynapse_e_syn", syn_e_syn)

# Calculate and update Rdl compartments
total_gaba_compartment_area_cm2 = 0.0
gaba_comp_geometry_cache = []

for b_idx, c_idx in gaba_compartments:
    branch_nodes = T4.branch(b_idx).nodes
    length_um = float(branch_nodes["length"].iloc[c_idx])
    radius_um = float(branch_nodes["radius"].iloc[c_idx])
    
    # Cylindrical surface area: 2 * pi * r * h (converted to cm^2)
    area_cm2 = (2 * np.pi * radius_um * length_um) * 1e-8
    total_gaba_compartment_area_cm2 += area_cm2
    gaba_comp_geometry_cache.append((b_idx, c_idx))

if total_gaba_compartment_area_cm2 > 0:
    # Compute the uniform density needed (S/cm^2)
    required_rdl_density = target_total_rdl_gS / total_gaba_compartment_area_cm2
    
    # Dial up the active locations to target density
    for b_idx, c_idx in gaba_comp_geometry_cache:
        T4.branch(b_idx).comp(c_idx).set("rdl_g", required_rdl_density)
        
    print("\nVoltage-dependent Rdl component:")
    print(f"  Target conductance: {target_total_rdl_gS * 1e6:.4f} μS")
    print(f"  Total active GABA shunt area: {total_gaba_compartment_area_cm2:.4e} cm²")
    print(f"  Calculated uniform density  : {required_rdl_density:.4e} S/cm²")

print("\nConnectivity summary:")
print(f" Connected Mi1 to {input_mapping['acetylcholine']['count']} dendritic ACh synapses.")
print(f" Connected Mi9 to {input_mapping['glutamate']['count']} dendritic Glu synapses.")
print(f" Connected Mi4 to {input_mapping['gaba']['count']} dendritic GABA synapses.")
    

In [ ]:
# Quick check of synapse locations in Jaxley
Mi9.move_to(40, 850)
Mi1.move_to(52, 850)
Mi4.move_to(65, 850)

fig, ax = plt.subplots(1, 1, figsize=(2, 3))
ax = net.vis(ax=ax)
ax.set_aspect("equal")
ax.set_axis_off()

In [ ]:
# # Interactive view for inspection
# pio.renderers.default = 'iframe'

def load_mesh(filepath):
    vertices, faces = [], []
    with open(filepath, 'r') as f:
        for line in f:
            if line.startswith('v '):
                vertices.append([float(p) for p in line.strip().split()[1:4]])
            elif line.startswith('f '):
                parts = line.strip().split()[1:]
                idx = [int(p.split('/')[0]) - 1 for p in parts]
                if len(idx) == 3:
                    faces.append(idx)
                elif len(idx) > 3:
                    for i in range(1, len(idx) - 1):
                        faces.append([idx[0], idx[i], idx[i+1]])
    return np.array(vertices), np.array(faces)

# Load mesh
obj_fname = inputpath_data / 'T4_full_mesh.obj'
raw_vertices, faces = load_mesh(obj_fname)

# Check if vertices are in nm (>1000) or already in µm
mesh_max = np.max(np.abs(raw_vertices))
obj_scale = 0.001 if mesh_max > 1000.0 else 1.0
vertices = raw_vertices * obj_scale

# Convert synapse coordinates to µm
if np.max(syn_x) > 10000.0:  # Raw FlyWire nm/voxel coordinates
    syn_x_um = syn_x * 0.001
    syn_y_um = syn_y * 0.001
    syn_z_um = syn_z * 0.001
else:
    syn_x_um = syn_x.copy()
    syn_y_um = syn_y.copy()
    syn_z_um = syn_z.copy()

# Map synapses
synapse_groups = {
    0: {"x": [], "y": [], "z": [], "name": "Glu", "color": "#117733"},
    1: {"x": [], "y": [], "z": [], "name": "ACh", "color": "#2B5C8F"},
    2: {"x": [], "y": [], "z": [], "name": "GABA", "color": "#44AA99"}
}

# Distance threshold (µm)
cutoff_dist_um = 5.0

for pre_idx, syn_type in [(0, "glutamate"), (1, "acetylcholine"), (2, "gaba")]:
    type_mask = (syn_types == syn_type)
    s_x = syn_x_um[type_mask]
    s_y = syn_y_um[type_mask]
    s_z = syn_z_um[type_mask]

    if len(s_x) > 0:
        # Distance to Jaxley comp_coords (which must also be in µm)
        dx = s_x[:, None] - comp_coords[None, :, 0]
        dy = s_y[:, None] - comp_coords[None, :, 1]
        dz = s_z[:, None] - comp_coords[None, :, 2]
        
        min_dists = np.min(np.sqrt(dx**2 + dy**2 + dz**2), axis=1)
        mapped_mask = min_dists <= cutoff_dist_um

        print(f"{syn_type.upper():12s}: Total={len(s_x)}, Min dist range=[{np.min(min_dists):.2f}, {np.max(min_dists):.2f}] µm; mapped={np.sum(mapped_mask)}")

        synapse_groups[pre_idx]["x"] = s_x[mapped_mask]
        synapse_groups[pre_idx]["y"] = s_y[mapped_mask]
        synapse_groups[pre_idx]["z"] = s_z[mapped_mask]


# Check cell size (in µm)
x_min, x_max = np.min(comp_coords[:, 0]), np.max(comp_coords[:, 0])
y_min, y_max = np.min(comp_coords[:, 1]), np.max(comp_coords[:, 1])
z_min, z_max = np.min(comp_coords[:, 2]), np.max(comp_coords[:, 2])

span_x = x_max - x_min
span_y = y_max - y_min
span_z = z_max - z_min

print("Cell dimensions:")
print(f"  X: {span_x:6.2f} µm")
print(f"  Y: {span_y:6.2f} µm")
print(f"  Z: {span_z:6.2f} µm")

# Render scene
fig = go.Figure()

# Add mesh
fig.add_trace(go.Mesh3d(
    x=vertices[:, 0], y=vertices[:, 1], z=vertices[:, 2],
    i=faces[:, 0], j=faces[:, 1], k=faces[:, 2],
    color='rgb(220, 220, 220)',
    opacity=0.30,
    name='T4 mesh',
    showlegend=True,
    flatshading=False,
    lighting=dict(ambient=0.5, diffuse=0.6, roughness=0.4)
))

# Add synapses
for pre_idx, group in synapse_groups.items():
    n_syn = len(group["x"])
    if n_syn > 0:
        fig.add_trace(go.Scatter3d(
            x=group["x"], y=group["y"], z=group["z"],
            mode='markers',
            marker=dict(
                size=4.0,
                color=group["color"],
                opacity=0.95,
                line=dict(color='black', width=0.8)
            ),
            name=f"{group['name']} ({n_syn} mapped)"
        ))

fig.update_layout(
    title="T4 dendritic input synapses",
    scene=dict(
        aspectmode='data',
        xaxis=dict(visible=False),
        yaxis=dict(visible=False),
        zaxis=dict(visible=False)
    ),
    margin=dict(l=0, r=0, b=0, t=40),
    legend=dict(yanchor="top", y=0.95, xanchor="left", x=0.05),
    template="plotly_white"
)

# Test simulations

In [ ]:
net.delete_recordings()

net.cell(range(0, 3)).record("v")
net.select(edges=[0, 1, 2]).record("ConductanceSynapse_s")

net.axon_terminal.branch(0).comp(0).record("v")
net.soma.branch(0).comp(0).record("v")

In [ ]:
amp = 0.02
t_max = 1800.0
dt = 0.025

time_vec = np.arange(0, t_max + 2 * dt, dt)

std = 200.0

distance = 5.0  # degree
duration = 5.0  # degree

speed = 25.0  # degree / s
speed /= 1000.0  # degree / ms

duration = duration / speed  # ms

def compute_currents(angle, two_arm_detector_delay = 200.0):
    dist_at_angle = jnp.cos(angle) * distance  # um
    delay = dist_at_angle / speed  # ms

    base = 600.0
    if delay > 0:
        delay1 = base
        delay2 = base + delay
        delay3 = base + 2 * delay
    else:
        delay1 = base - 2 * delay
        delay2 = base - delay
        delay3 = base
    
    # HRBL detector with a delay on the first and third arm
    delay1 += two_arm_detector_delay
    delay3 += two_arm_detector_delay
    
    current1 = jnp.exp(-((time_vec - delay1) ** 2) / std**2) * amp * (-1) + amp
    current2 = jnp.exp(-((time_vec - delay2) ** 2) / std**2) * amp 
    current3 = jnp.exp(-((time_vec - delay3) ** 2) / std**2) * amp

    return current1, current2, current3

In [ ]:
with open(savepath_data / 'pd_stim.pkl', 'wb') as handle:
    pickle.dump(compute_currents(0.0, 0.0), handle)
    
with open(savepath_data / 'orth_stim.pkl', 'wb') as handle:
    pickle.dump(compute_currents(jnp.pi, 0.0), handle)

In [ ]:
def simulate(angle):
    current1, current2, current3 = compute_currents(angle)
    data_stim = None
    data_stim = Mi9.data_stimulate(current1[:-1], data_stimuli=data_stim)
    data_stim = Mi1.data_stimulate(current2[:-1], data_stimuli=data_stim)
    data_stim = Mi4.data_stimulate(current3[:-1], data_stimuli=data_stim)
    return jx.integrate(net, data_stimuli=data_stim)

In [ ]:
# Change input resistance of the input cells, but not the T4 neuron
net.cell([0, 1, 2]).set("Leak_gLeak", 1 / 5_000)

# Change time constant of Mi4 and Mi9 (by increasing the capacitance)
net.cell(2).set("capacitance", 50.0) # Mi4
net.cell(0).set("capacitance", 10.0) # Mi9

net.set("Leak_eLeak", -60)
T4.set("Leak_eLeak", 0)

net.set("ConductanceSynapse_v_th", -57.0)
net.set("ConductanceSynapse_delta", 2.0)

# Set initial resting potentials
net.set("v", -60)              # T4
net.cell([1, 2]).set("v", -60) # Mi1 and Mi4
Mi9.set("v", -44)              # Mi9 tonically depolarized for disinhibition

net.delete_recordings()
net.cell(range(0, 3)).record("v")
net.select(edges=[0, 1, 2]).record("ConductanceSynapse_s")

net.axon_terminal.branch(0).comp(0).record("v")
net.soma.branch(0).comp(0).record("v")


In [ ]:
# PD
current1, current2, current3 = compute_currents(0.0)
v = simulate(0.0)

# Plot
fig, ax = plt.subplots(1, 3, figsize=(12, 3))
ax[0].plot(time_vec, current1, color="#117733", alpha=0.8, lw=2, label="Mi9 (Glu)")
ax[0].plot(time_vec, current2, color="#332288", alpha=0.8, lw=2, label="Mi1 (ACh)")
ax[0].plot(time_vec, current3, color="#44AA99", alpha=0.8, lw=2, label="Mi4 (GABA)")
ax[0].set_title("Input signals")
ax[0].set_ylabel("Current (nA)")
ax[0].legend(frameon=False)

ax[1].plot(time_vec, v[0], color="#117733", alpha=0.8, lw=2, label="Mi9 (Glu)")
ax[1].plot(time_vec, v[1], color="#332288", alpha=0.8, lw=2, label="Mi1 (ACh)")
ax[1].plot(time_vec, v[2], color="#44AA99", alpha=0.8, lw=2, label="Mi4 (GABA)")
ax[1].set_ylim([-65, -35])
ax[1].set_title("Presynaptic membrane potentials")

ax[2].plot(time_vec, v[6], color="k", clip_on=False, lw=2, label="T4 Post")
ax[2].set_ylim([-65, -35])
ax[2].set_title("T4 membrane potential")
ax[2].set_ylabel("Membrane potential (mV)")

for a in ax:
    a.spines[['right', 'top']].set_visible(False)
    a.set_xlabel("Time (ms)")

ax[1].set_ylabel("Membrane potential (mV)")

plt.tight_layout()

with open(savepath_data / 'pd.pkl', "wb") as handle:
    pickle.dump(v[:8], handle)

In [ ]:
# ND
current1, current2, current3 = compute_currents(pi)
v = simulate(pi)

# Plot
fig, ax = plt.subplots(1, 3, figsize=(12, 3))
ax[0].plot(time_vec, current1, color="#117733", alpha=0.8, lw=2, label="Mi9 (Glu)")
ax[0].plot(time_vec, current2, color="#332288", alpha=0.8, lw=2, label="Mi1 (ACh)")
ax[0].plot(time_vec, current3, color="#44AA99", alpha=0.8, lw=2, label="Mi4 (GABA)")
ax[0].set_title("Input signals")
ax[0].set_ylabel("Current (nA)")
ax[0].legend(frameon=False)

ax[1].plot(time_vec, v[0], color="#117733", alpha=0.8, lw=2, label="Mi9 (Glu)")
ax[1].plot(time_vec, v[1], color="#332288", alpha=0.8, lw=2, label="Mi1 (ACh)")
ax[1].plot(time_vec, v[2], color="#44AA99", alpha=0.8, lw=2, label="Mi4 (GABA)")
ax[1].set_ylim([-65, -35])
ax[1].set_title("Presynaptic membrane potentials")
#ax[1].legend(frameon=False)

ax[2].plot(time_vec, v[6], color="k", clip_on=False, lw=2, label="T4 Post")
ax[2].set_ylim([-65, -35])
ax[2].set_title("T4 membrane potential")
ax[2].set_ylabel("Membrane potential (mV)")

for a in ax:
    a.spines[['right', 'top']].set_visible(False)
    a.set_xlabel("Time (ms)")

ax[1].set_ylabel("Membrane potential (mV)")
plt.tight_layout()

with open(savepath_data / 'orth.pkl', "wb") as handle:
    pickle.dump(v[:8], handle)

In [ ]:
def _run_sweep(net, T4, simulate_fn, num_directions=20):
    net.delete_recordings()
    try:
        net.axon_terminal.branch(0).comp(0).record("v")
    except AttributeError:
        T4.branch(0).comp(0).record("v")
    
    angles = np.linspace(0, 2 * np.pi, num_directions + 1) # 20 + 1 directions (18° steps)
    v_traces = np.asarray([simulate_fn(angle)[0] for angle in angles])
    return v_traces, angles


def run_simulations(net, T4, gaba_edges, simulate_fn, gaba_comp_cache, required_rdl_density, time_vec=None, filename=savepath_data / "t4_simulation_data.pkl"):
    
    # Save original GABA synapse conductances
    gaba_synapses = net.select(edges=gaba_edges)
    orig_gaba_gS = gaba_synapses.edges["ConductanceSynapse_gS"].to_numpy()

    # Set rdl_g on the GABA synapse compartments
    def set_rdl_density(density_value):
        for b_idx, c_idx in gaba_comp_cache:
            T4.branch(b_idx).comp(c_idx).set("rdl_g", density_value)

    try:
        print("1) Simulating full model...")
        set_rdl_density(required_rdl_density)
        gaba_synapses.set("ConductanceSynapse_gS", orig_gaba_gS)
        v_full, angles_rad = _run_sweep(net, T4, simulate_fn, num_directions=20)

        print("2) Simulating without voltage-dependence of Rdl...")
        set_rdl_density(0.0)
        v_no_volt_rdl, _ = _run_sweep(net, T4, simulate_fn, num_directions=20)

        print("3) Simulating without Rdl...")
        set_rdl_density(0.0)
        gaba_synapses.set("ConductanceSynapse_gS", 0.0)
        v_no_rdl, _ = _run_sweep(net, T4, simulate_fn, num_directions=20)

    finally:
        # Restore original state
        set_rdl_density(required_rdl_density)
        gaba_synapses.set("ConductanceSynapse_gS", orig_gaba_gS)

    if time_vec is None:
        time_vec = np.arange(v_full.shape[1])

    sim_data = {
        "v_full": v_full,
        "v_no_volt_rdl": v_no_volt_rdl,
        "v_no_rdl": v_no_rdl,
        "angles_rad": angles_rad,
        "time_vec": time_vec
    }

    with open(filename, "wb") as f:
        pickle.dump(sim_data, f)

    print(f"Data saved to '{filename}'.")
    return filename

In [ ]:
run_simulations(
    net=net,
    T4=T4,
    gaba_edges=wired_edge_indices_by_type["gaba"],
    simulate_fn=simulate,
    gaba_comp_cache=gaba_comp_geometry_cache,
    required_rdl_density=required_rdl_density,
    time_vec=time_vec
)

In [ ]:
cm_inch = 1/2.54 

def get_mean_around_peak(v_matrix, window_radius):
    n_dirs, n_pts = v_matrix.shape
    peak_means = np.zeros(n_dirs)

    for i in range(n_dirs):
        peak_idx = np.argmax(v_matrix[i])
    
        start_idx = max(0, peak_idx - window_radius)
        end_idx = min(n_pts, peak_idx + window_radius + 1)
        
        peak_means[i] = np.mean(v_matrix[i, start_idx:end_idx])

    return peak_means

def plot_simulations(pickle_path=savepath_data / "t4_simulation_data.pkl", resp='max', save_pdf=True):
    
    # Load data
    with open(pickle_path, "rb") as f:
        sim_data = pickle.load(f)

    v_full_raw = sim_data["v_full"]
    v_no_volt_rdl_raw = sim_data["v_no_volt_rdl"]
    v_no_rdl_raw = sim_data["v_no_rdl"]
    angles_rad_full = sim_data["angles_rad"]
    time_vec_raw = sim_data["time_vec"]

    clip_pts = 1000  # Clip initial Vm values
    v_full = v_full_raw[:, clip_pts:]
    v_no_volt_rdl = v_no_volt_rdl_raw[:, clip_pts:]
    v_no_rdl = v_no_rdl_raw[:, clip_pts:]
    time_vec = time_vec_raw[clip_pts:]
    t_max = time_vec[-1]

    # Baseline membrane potential
    base_full = np.mean(v_full[:, 1000:1100], axis=1, keepdims=True)
    base_no_volt = np.mean(v_no_volt_rdl[:, 1000:1100], axis=1, keepdims=True)
    base_no_rdl = np.mean(v_no_rdl[:, 1000:1100], axis=1, keepdims=True)

    # Minimum membrane potential
    min_full = np.min(v_full, axis=1, keepdims=True)
    min_no_volt = np.min(v_no_volt_rdl, axis=1, keepdims=True)
    min_no_rdl = np.min(v_no_rdl, axis=1, keepdims=True)
    
    if resp == 'max':
        # Get peak amplitudes
        resp_full_raw = np.max(v_full - min_full, axis=1)
        resp_no_volt_rdl_raw = np.max(v_no_volt_rdl - min_no_volt, axis=1)
        resp_no_rdl_raw = np.max(v_no_rdl - min_no_rdl, axis=1)

    if resp == 'mean':
        # Get mean values
        resp_full_raw = np.mean(v_full - base_full, axis=1)
        resp_no_volt_rdl_raw = np.mean(v_no_volt_rdl - base_no_volt, axis=1)
        resp_no_rdl_raw = np.mean(v_no_rdl - base_no_rdl, axis=1)

    if resp == 'mean_peak':
        # Get mean values around the peak
        v_full_sub = v_full - base_full
        v_no_volt_sub = v_no_volt_rdl - base_no_volt
        v_no_rdl_sub = v_no_rdl - base_no_rdl

        window_radius = 3000 # +/- 75 ms
        
        resp_full_raw = get_mean_around_peak(v_full_sub, window_radius=window_radius)
        resp_no_volt_rdl_raw = get_mean_around_peak(v_no_volt_sub, window_radius=window_radius)
        resp_no_rdl_raw = get_mean_around_peak(v_no_rdl_sub, window_radius=window_radius)
        
    # Normalize each model individually
    norm_full = resp_full_raw[0] if resp_full_raw[0] != 0 else 1.0
    norm_no_volt = resp_no_volt_rdl_raw[0] if resp_no_volt_rdl_raw[0] != 0 else 1.0
    norm_no_rdl = resp_no_rdl_raw[0] if resp_no_rdl_raw[0] != 0 else 1.0

    resp_full_norm = resp_full_raw / norm_full
    resp_no_volt_rdl_norm = resp_no_volt_rdl_raw / norm_no_volt
    resp_no_rdl_norm = resp_no_rdl_raw / norm_no_rdl

    # 20 directions for traces
    col_angles = np.linspace(-180, 162, 20)

    FONT_SIZE = 6
    fig = plt.figure(figsize=(10*cm_inch, 10*cm_inch))
    gs = fig.add_gridspec(2, 1, height_ratios=[0.8,1], hspace=0.25)
    gs_traces = gs[0].subgridspec(3, 20, wspace=0.15, hspace=0.25)
    gs_polar = gs[1].subgridspec(1, 3, wspace=0.4)

    conditions = [
        {"name": "Full model", "data": v_full, "color": "#000000", "polar_data": resp_full_norm},
        {"name": "No voltage-dependent Rdl", "data": v_no_volt_rdl, "color": "#416299", "polar_data": resp_no_volt_rdl_norm},
        {"name": "No Rdl", "data": v_no_rdl, "color": "#52c3c3", "polar_data": resp_no_rdl_norm}
    ]

    all_voltages = np.concatenate([v_full, v_no_volt_rdl, v_no_rdl])
    v_min, v_max = np.min(all_voltages) - 1.0, np.max(all_voltages) + 1.0

    # Traces
    axes_traces = np.empty((3, 20), dtype=object)
    for row_idx, cond in enumerate(conditions):
        data = cond["data"]
        color = cond["color"]

        for col_idx in range(20):
            ax = fig.add_subplot(gs_traces[row_idx, col_idx])
            axes_traces[row_idx, col_idx] = ax
            ax.axis('off')
            ax.set_ylim([v_min, v_max])

            sim_idx = (col_idx + 10) % 20
            ax.plot(time_vec, data[sim_idx], color=color, lw=1.2)

            if col_idx == 0:
                ax.text(
                    -0.3, 0.5, cond["name"], transform=ax.transAxes,
                    rotation=90, ha='right', va='center', fontsize=FONT_SIZE,
                    fontweight='bold', color=color
                )

            if row_idx == 0:
                ax.set_title(
                    f"{col_angles[col_idx]:.0f}°",
                    fontsize=FONT_SIZE, pad=8, color="black", fontweight="normal"
                )

    # Scale bar
    scale_ax = axes_traces[2, 0]
    t_start = time_vec[0] + (t_max - time_vec[0]) * 0.05
    v_start = v_min + (v_max - v_min) * 0.05

    v_len = 10.0      # 10 mV
    t_len = 1000.0    # 1000 ms

    scale_ax.plot([t_start, t_start + t_len], [v_start, v_start], color="black", lw=0.75)
    scale_ax.text(t_start + t_len / 2, v_start - (v_max - v_min) * 0.08, "1 s",
                  ha="center", va="top", fontsize=FONT_SIZE, fontweight=700)

    scale_ax.plot([t_start, t_start], [v_start, v_start + v_len], color="black", lw=0.75)
    scale_ax.text(t_start - (t_max - time_vec[0]) * 0.05, v_start + v_len / 2, "10 mV",
                  ha="right", va="center", fontsize=FONT_SIZE, rotation=90)

    # Polar plots
    for col_idx, cond in enumerate(conditions):
        ax = fig.add_subplot(gs_polar[0, col_idx], projection='polar')
        color = cond["color"]

        ax.plot(angles_rad_full, cond["polar_data"], color=color, lw=1)
        ax.fill(angles_rad_full, cond["polar_data"], color=color, alpha=0.15)

        ax.set_theta_zero_location("E")
        ax.set_theta_direction(1)
        ax.set_ylim(0, 1.1)

        ax.set_rticks([0.25, 0.5, 0.75, 1.0])

        tick_angles_deg = [0, 45, 90, 135, 180, 225, 270, 315]
        ax.set_xticks(np.deg2rad(tick_angles_deg))
        ax.set_xticklabels('')#[f"{d}°" for d in tick_angles_deg], fontsize=FONT_SIZE)

        ax.tick_params(axis='both', labelsize=FONT_SIZE, colors="#555555")
        ax.grid(True, color="k", alpha=0.3)
        ax.grid(linewidth=0.5, linestyle='dotted')
        ax.set_title(cond["name"], fontsize=FONT_SIZE, fontweight="bold", color=color, pad=15)

    if save_pdf:
        plt.savefig(savepath_fig / 'model_rdl.pdf')

In [ ]:
plot_simulations(resp='mean_peak')